# 🛡️ DeepFake Shield — Exploratory Analysis Notebook

This notebook covers:
1. Dataset exploration
2. Model architecture inspection
3. Training results analysis
4. Grad-CAM visualization
5. Single-video inference demo

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0a0e1a'
matplotlib.rcParams['axes.facecolor']   = '#0a0e1a'
matplotlib.rcParams['text.color']       = 'white'

print('✅ Imports OK')

## 1. Build & Inspect Model

In [ ]:
from app.models.detector import build_deepfake_detector, build_lightweight_model

# Build full model
model = build_deepfake_detector()
model.summary()

print(f'\nTotal params: {model.count_params():,}')

## 2. Single Video Analysis

In [ ]:
from app.models.predictor import DeepFakePredictor

predictor = DeepFakePredictor(lightweight=True)

# Replace with your video path
VIDEO_PATH = '../dataset/fake/sample.mp4'

result = predictor.predict(VIDEO_PATH)

print(f"Verdict:    {result['verdict']}")
print(f"Confidence: {result['confidence']:.1%}")
print(f"Fake Score: {result['fake_score']:.4f}")
print(f"Frames:     {result['total_frames']}")
print(f"Faces:      {result['frames_with_faces']}")
print(f"Time:       {result['processing_time_sec']}s")

## 3. Visualize Extracted Faces

In [ ]:
import cv2

face_images = result['face_images'][:12]
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
fig.suptitle('Extracted Face Crops', color='white', fontsize=14)

for i, (ax, face) in enumerate(zip(axes.flat, face_images)):
    rgb = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
    ax.imshow(rgb)
    ax.axis('off')
    ax.set_title(f'Frame {i}', color='#80deea', fontsize=9)

plt.tight_layout()
plt.show()

## 4. Frame Score Timeline

In [ ]:
frame_scores = result['frame_scores']

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(frame_scores, color='#ff4757', linewidth=2, label='Fake Probability')
ax.axhline(0.5, color='white', linestyle='--', alpha=0.5, label='Threshold')
ax.fill_between(range(len(frame_scores)), frame_scores, alpha=0.3, color='#ff4757')
ax.set_xlabel('Frame Index', color='white')
ax.set_ylabel('Fake Probability', color='white')
ax.set_title('Per-Frame Fake Score Timeline', color='white')
ax.set_ylim(0, 1)
ax.legend()
ax.tick_params(colors='white')
plt.tight_layout()
plt.show()

## 5. Load Training History (if available)

In [ ]:
import json
from pathlib import Path

history_file = Path('../logs/training_history.json')

if history_file.exists():
    with open(history_file) as f:
        data = json.load(f)
    
    history = data['history']
    epochs = range(1, len(history['loss']) + 1)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(epochs, history['loss'],     label='Train Loss', color='#80deea')
    axes[0].plot(epochs, history['val_loss'], label='Val Loss',   color='#ff4757')
    axes[0].set_title('Loss', color='white')
    axes[0].legend()
    axes[0].tick_params(colors='white')
    
    axes[1].plot(epochs, history['auc'],     label='Train AUC', color='#00e676')
    axes[1].plot(epochs, history['val_auc'], label='Val AUC',   color='#ffd740')
    axes[1].set_title('AUC', color='white')
    axes[1].legend()
    axes[1].tick_params(colors='white')
    
    plt.suptitle('Training History', color='white', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    print('\nFinal Metrics:')
    for k, v in data.get('metrics', {}).items():
        if k != 'confusion_matrix':
            print(f'  {k}: {v:.4f}')
else:
    print('No training history found. Run train.py first.')